# PLOS revision — Drug-grouped Genotype-only CNN

This notebook updates the previous genotype-only `SimpleCNN` baseline for the revised validation design.

### Main changes
- Uses the full `dataset.csv`
- **Outer 5-fold StratifiedGroupKFold grouped by Drug**
- Within each outer-training partition, a **Drug-grouped inner train/validation split**
- Train / validation / outer-test drugs are mutually disjoint
- Validation **Macro-F1** is used for checkpoint selection / early stopping
- The untouched outer test fold is evaluated only after selecting the best checkpoint
- Reports Accuracy, Macro-Precision, Macro-Recall, Macro-F1, Macro-AUROC (OvR), Macro-AUPR (OvR)
- Saves fold assignments, fold-level results, summary statistics, out-of-fold predictions, ROC/PR curves, and a full confusion matrix
- No genotype augmentation is applied

The split seeds and training settings are aligned with the no-augmentation multimodal Drug-grouped experiment so that the comparison is paired by fold.

In [23]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_curve, auc, precision_recall_curve, roc_auc_score,
    average_precision_score, confusion_matrix
)
import matplotlib.pyplot as plt

SEED = 2
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


Device: cuda:1


In [24]:
# ================================
# 1. Load full dataset
# ================================
DATA_FILE = 'dataset.csv'
data = pd.read_csv(DATA_FILE)

required_cols = {'Variant', 'Drug', 'Class'}
missing = required_cols - set(data.columns)
if missing:
    raise ValueError(f'Missing required columns: {missing}')

label_encoder = LabelEncoder()
data['Class_encoded'] = label_encoder.fit_transform(data['Class'])
class_mapping = {i: label for i, label in enumerate(label_encoder.classes_)}

print('Total samples:', len(data))
print('Number of drugs:', data['Drug'].nunique())
print('Class mapping:', class_mapping)
print('\nClass counts:')
print(data['Class'].value_counts())


Total samples: 622
Number of drugs: 59
Class mapping: {0: 'de', 1: 'f', 2: 'nor'}

Class counts:
de     250
f      195
nor    177
Name: Class, dtype: int64


In [25]:
# ================================
# 2. Load CYP2D6 one-hot matrices
# ================================
ONEHOT_FOLDER = 'onehot_7'
onehot_files = os.listdir(ONEHOT_FOLDER)
onehot_matrices = {}
max_rows = 0

for file in onehot_files:
    if not file.endswith('.csv'):
        continue

    variant_name = file.split('.csv')[0].replace('_', '.')
    file_path = os.path.join(ONEHOT_FOLDER, file)

    try:
        matrix = pd.read_csv(file_path, header=None).values
        onehot_matrices[variant_name] = matrix
        max_rows = max(max_rows, matrix.shape[0])
    except Exception as e:
        print(f'Error loading {file}: {e}')

print('Maximum rows:', max_rows)
print('Loaded matrices:', len(onehot_matrices))


Maximum rows: 4330
Loaded matrices: 361


In [26]:
# ================================
# 3. Variant -> CNN tensor
#    Preserves the original genotype-only representation:
#    two 4330x7 haplotype matrices are vertically concatenated
#    -> 8660x7
# ================================
TARGET_ROWS = 4330
TARGET_COLS = 7

def pad_matrix(matrix, target_rows=TARGET_ROWS, target_cols=TARGET_COLS):
    current_rows, current_cols = matrix.shape
    if current_rows > target_rows or current_cols > target_cols:
        raise ValueError(
            f'Matrix shape {matrix.shape} exceeds target {(target_rows, target_cols)}'
        )
    padded = torch.zeros((target_rows, target_cols), dtype=torch.float32)
    padded[:current_rows, :current_cols] = torch.tensor(matrix, dtype=torch.float32)
    return padded

for key in list(onehot_matrices.keys()):
    onehot_matrices[key] = pad_matrix(onehot_matrices[key])

def split_diplotype(variant):
    # Supports both "A; B" and "A;B"
    return [v.strip() for v in str(variant).split(';') if v.strip()]

def map_variant_to_onehot(variant):
    variants = split_diplotype(variant)
    matrices = []

    for v in variants[:2]:
        if v not in onehot_matrices:
            raise KeyError(
                f'One-hot matrix not found for variant: {v}\n'
                'Check Variant nomenclature and onehot_7 filenames.'
            )
        matrices.append(onehot_matrices[v])

    while len(matrices) < 2:
        matrices.append(torch.zeros((TARGET_ROWS, TARGET_COLS), dtype=torch.float32))

    return torch.vstack(matrices[:2])  # (8660, 7)

def map_variants_to_tensor(df):
    X = torch.stack([map_variant_to_onehot(v) for v in df['Variant']])
    y = torch.tensor(df['Class_encoded'].values, dtype=torch.long)
    return X, y

# Fail early if any allele cannot be mapped
all_X, all_y = map_variants_to_tensor(data)
print('Full genotype tensor:', all_X.shape)
print('Full labels:', all_y.shape)


Full genotype tensor: torch.Size([622, 8660, 7])
Full labels: torch.Size([622])


In [27]:
# ================================
# 4. Dataset
# ================================
class GenotypeDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        # X[idx]: (8660, 7) -> (1, 8660, 7)
        return self.X[idx].float().unsqueeze(0), self.y[idx].long()

BATCH_SIZE = 32


## CNN architecture

The architecture below preserves the previous genotype-only model:
`Conv -> Pool -> Conv -> Pool -> FC(128) -> 3-class output`.

This is intentionally kept close to the original single-CNN baseline rather than redesigning the model during the validation revision.

In [28]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=(20, 7), stride=(1, 1))
        self.pool1 = nn.MaxPool2d(kernel_size=(2, 1))  # 추가된 풀링 층
        self.conv2 = nn.Conv2d(in_channels=8, out_channels=16, kernel_size=(20, 1), stride=(1, 1))
        self.pool2 = nn.MaxPool2d(kernel_size=(2, 1))  # 추가된 풀링 층
        
        # 풀링 층 추가로 인한 출력 크기 변경을 고려한 새로운 계산
        self.fc1 = nn.Linear(16 * ((((8660 - 20) // 2) - 20) // 2), 128)
        self.fc2 = nn.Linear(128, 3)  # 출력 층

    def forward(self, x):
        x = self.conv1(x)
        x = nn.ReLU()(x)
        x = self.pool1(x)  # 풀링 적용
        x = self.conv2(x)
        x = nn.ReLU()(x)
        x = self.pool2(x)  # 풀링 적용
        x = x.view(x.size(0), -1)  # 평탄화
        x = self.fc1(x)
        x = nn.ReLU()(x)
        x = self.fc2(x)
        return x

# Shape sanity check
_tmp = SimpleCNN().to(device)
with torch.no_grad():
    _out = _tmp(torch.zeros(2, 1, 8660, 7, device=device))
print('CNN output shape:', tuple(_out.shape))
del _tmp, _out
if torch.cuda.is_available():
    torch.cuda.empty_cache()


CNN output shape: (2, 3)


In [29]:
# ================================
# 5. Metrics
# ================================
def compute_macro_aupr(y_true, y_probs, num_classes=3):
    values = []
    for i in range(num_classes):
        y_binary = (y_true == i).astype(int)
        if y_binary.sum() == 0:
            values.append(np.nan)
        else:
            values.append(
                average_precision_score(y_binary, y_probs[:, i])
            )
    return float(np.nanmean(values))

def compute_macro_auc(y_true, y_probs, num_classes=3):
    values = []
    for i in range(num_classes):
        y_binary = (y_true == i).astype(int)
        pos = y_binary.sum()
        neg = len(y_binary) - pos

        if pos == 0 or neg == 0:
            values.append(np.nan)
        else:
            values.append(
                roc_auc_score(y_binary, y_probs[:, i])
            )
    return float(np.nanmean(values))

def evaluate_model(model, loader, criterion, device, num_classes=3):
    model.eval()

    total_loss = 0.0
    all_labels = []
    all_logits = []

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            # sample-weighted loss
            total_loss += loss.item() * labels.size(0)
            all_labels.append(labels.cpu())
            all_logits.append(outputs.cpu())

    y_true = torch.cat(all_labels).numpy()
    logits = torch.cat(all_logits)
    y_probs = F.softmax(logits, dim=1).numpy()
    y_pred = y_probs.argmax(axis=1)

    return {
        'loss': total_loss / len(y_true),
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(
            y_true, y_pred, average='macro', zero_division=0
        ),
        'recall': recall_score(
            y_true, y_pred, average='macro', zero_division=0
        ),
        'f1': f1_score(
            y_true, y_pred, average='macro', zero_division=0
        ),
        'auc': compute_macro_auc(y_true, y_probs, num_classes),
        'aupr': compute_macro_aupr(y_true, y_probs, num_classes),
        'y_true': y_true,
        'y_pred': y_pred,
        'y_probs': y_probs
    }


In [30]:
# ================================
# 6. Drug-grouped outer/inner splits
#    Same split logic/seeds as multimodal ×1 experiment
# ================================
N_OUTER_SPLITS = 5
N_INNER_SPLITS = 10
TARGET_VAL_N = 80

outer_cv = StratifiedGroupKFold(
    n_splits=N_OUTER_SPLITS,
    shuffle=True,
    random_state=SEED
)

outer_splits = list(
    outer_cv.split(
        X=np.zeros(len(data)),
        y=data['Class_encoded'].values,
        groups=data['Drug'].astype(str).values
    )
)


def make_inner_train_val_split(
    outer_train_idx,
    df,
    random_state,
    target_val_n=TARGET_VAL_N
):
    outer_train_df = df.iloc[outer_train_idx]

    inner_cv = StratifiedGroupKFold(
        n_splits=N_INNER_SPLITS,
        shuffle=True,
        random_state=random_state
    )

    best_split = None
    best_diff = float('inf')

    for inner_train_local, val_local in inner_cv.split(
        X=np.zeros(len(outer_train_df)),
        y=outer_train_df['Class_encoded'].values,
        groups=outer_train_df['Drug'].astype(str).values
    ):
        diff = abs(len(val_local) - target_val_n)

        if diff < best_diff:
            best_diff = diff
            best_split = (
                inner_train_local,
                val_local
            )

    inner_train_local, val_local = best_split

    return (
        outer_train_idx[inner_train_local],
        outer_train_idx[val_local]
    )
print('Outer folds:', len(outer_splits))

# Show split sizes and verify Drug disjointness
for fold, (outer_train_idx, test_idx) in enumerate(outer_splits, start=1):
    train_idx, val_idx = make_inner_train_val_split(
        outer_train_idx, data, random_state=100 + (fold - 1)
    )

    train_drugs = set(data.iloc[train_idx]['Drug'].astype(str))
    val_drugs = set(data.iloc[val_idx]['Drug'].astype(str))
    test_drugs = set(data.iloc[test_idx]['Drug'].astype(str))

    assert train_drugs.isdisjoint(val_drugs)
    assert train_drugs.isdisjoint(test_drugs)
    assert val_drugs.isdisjoint(test_drugs)

    print(
        f'Fold {fold}: '
        f'train={len(train_idx)} ({len(train_drugs)} drugs), '
        f'val={len(val_idx)} ({len(val_drugs)} drugs), '
        f'test={len(test_idx)} ({len(test_drugs)} drugs)'
    )


Outer folds: 5
Fold 1: train=454 (43 drugs), val=74 (4 drugs), test=94 (12 drugs)
Fold 2: train=425 (42 drugs), val=78 (6 drugs), test=119 (11 drugs)
Fold 3: train=394 (39 drugs), val=73 (8 drugs), test=155 (12 drugs)
Fold 4: train=431 (42 drugs), val=83 (6 drugs), test=108 (11 drugs)
Fold 5: train=396 (41 drugs), val=80 (5 drugs), test=146 (13 drugs)


## Training settings

These are deliberately aligned with the multimodal ×1 Drug-grouped experiment:
- Adam, LR = 0.001
- Batch size = 32
- Maximum 100 epochs
- Patience = 20
- Early stopping / checkpoint selection by **validation Macro-F1**

If you later change epoch/patience for the multimodal model, change them identically here before making the final paired comparison.

In [31]:
# ================================
# 7. Drug-grouped 5-fold experiment
# ================================
NUM_EPOCHS = 150
PATIENCE = 30
LEARNING_RATE = 0.001

criterion = nn.CrossEntropyLoss().to(device)

fold_results = []
fold_assignments = []
all_test_labels = []
all_test_probs = []

for fold, (outer_train_idx, test_idx) in enumerate(outer_splits):
    print('\n' + '=' * 90)
    print(f'OUTER FOLD {fold + 1}/{N_OUTER_SPLITS}')
    print('=' * 90)

    train_idx, val_idx = make_inner_train_val_split(
        outer_train_idx, data, random_state=100 + fold
    )

    train_df = data.iloc[train_idx].copy()
    val_df = data.iloc[val_idx].copy()
    test_df = data.iloc[test_idx].copy()

    train_drugs = set(train_df['Drug'].astype(str))
    val_drugs = set(val_df['Drug'].astype(str))
    test_drugs = set(test_df['Drug'].astype(str))

    assert train_drugs.isdisjoint(val_drugs)
    assert train_drugs.isdisjoint(test_drugs)
    assert val_drugs.isdisjoint(test_drugs)

    print(f'Train      : {len(train_df)} samples / {len(train_drugs)} drugs')
    print(f'Validation : {len(val_df)} samples / {len(val_drugs)} drugs')
    print(f'Outer Test : {len(test_df)} samples / {len(test_drugs)} drugs')
    print('Test drugs:', sorted(test_drugs))
    print('Train class counts:', train_df['Class'].value_counts().to_dict())
    print('Val class counts  :', val_df['Class'].value_counts().to_dict())
    print('Test class counts :', test_df['Class'].value_counts().to_dict())

    # Record exact split assignments
    for split_name, indices in [
        ('train', train_idx),
        ('validation', val_idx),
        ('test', test_idx)
    ]:
        for idx in indices:
            fold_assignments.append({
                'sample_index': int(idx),
                'outer_fold': fold + 1,
                'set': split_name,
                'Drug': data.iloc[idx]['Drug'],
                'Variant': data.iloc[idx]['Variant'],
                'Class': data.iloc[idx]['Class']
            })

    # Use precomputed full tensor so all factors/models map exactly the same samples
    X_train = all_X[train_idx]
    y_train = all_y[train_idx]
    X_val = all_X[val_idx]
    y_val = all_y[val_idx]
    X_test = all_X[test_idx]
    y_test = all_y[test_idx]

    train_dataset = GenotypeDataset(X_train, y_train)
    val_dataset = GenotypeDataset(X_val, y_val)
    test_dataset = GenotypeDataset(X_test, y_test)

    # Same per-fold initialization seed as multimodal experiment
    model_seed = 1000 + fold
    random.seed(model_seed)
    np.random.seed(model_seed)
    torch.manual_seed(model_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(model_seed)

    train_generator = torch.Generator()
    train_generator.manual_seed(model_seed)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=train_generator
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    model = SimpleCNN().to(device)
    optimizer = torch.optim.Adam(
        model.parameters(), lr=LEARNING_RATE
    )

    best_val_f1 = -np.inf
    best_epoch = None
    patience_counter = 0
    checkpoint_path = f'best_genotype_only_drug_grouped_fold_{fold + 1}.pth'

    history = []

    for epoch in range(NUM_EPOCHS):
        model.train()
        total_train_loss = 0.0
        total_train = 0
        correct_train = 0

        for inputs, labels in train_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item() * labels.size(0)
            total_train += labels.size(0)
            correct_train += (
                outputs.argmax(dim=1) == labels
            ).sum().item()

        train_loss = total_train_loss / total_train
        train_acc = correct_train / total_train

        val_metrics = evaluate_model(
            model, val_loader, criterion, device
        )

        history.append({
            'fold': fold + 1,
            'epoch': epoch + 1,
            'train_loss': train_loss,
            'train_accuracy': train_acc,
            'val_loss': val_metrics['loss'],
            'val_accuracy': val_metrics['accuracy'],
            'val_f1': val_metrics['f1'],
            'val_auc': val_metrics['auc'],
            'val_aupr': val_metrics['aupr']
        })

        print(
            f'Epoch [{epoch + 1:02d}/{NUM_EPOCHS}] | '
            f'Train Loss {train_loss:.4f} Acc {train_acc:.4f} | '
            f'Val Loss {val_metrics["loss"]:.4f} '
            f'Acc {val_metrics["accuracy"]:.4f} | '
            f'Val Macro-F1 {val_metrics["f1"]:.4f} '
            f'AUC {val_metrics["auc"]:.4f} '
            f'AUPR {val_metrics["aupr"]:.4f}'
        )

        # Best checkpoint by validation Macro-F1
        if val_metrics['f1'] > best_val_f1:
            best_val_f1 = val_metrics['f1']
            best_epoch = epoch + 1
            patience_counter = 0
            torch.save(model.state_dict(), checkpoint_path)
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print('Early stopping applied.')
                break

    # Save this fold's learning history
    pd.DataFrame(history).to_csv(
        f'genotype_only_training_history_fold_{fold + 1}.csv',
        index=False
    )

    # Reload BEST checkpoint before validation/test reporting
    model.load_state_dict(
        torch.load(checkpoint_path, map_location=device)
    )
    model.eval()

    best_val_metrics = evaluate_model(
        model, val_loader, criterion, device
    )
    test_metrics = evaluate_model(
        model, test_loader, criterion, device
    )

    print(f'Best epoch: {best_epoch}')
    print(f'Best Val Macro-F1: {best_val_metrics["f1"]:.4f}')
    print('*** OUTER TEST — UNSEEN DRUGS ***')
    print(
        f'Acc {test_metrics["accuracy"]:.4f} | '
        f'Macro-Precision {test_metrics["precision"]:.4f} | '
        f'Macro-Recall {test_metrics["recall"]:.4f} | '
        f'Macro-F1 {test_metrics["f1"]:.4f} | '
        f'Macro-AUROC {test_metrics["auc"]:.4f} | '
        f'Macro-AUPR {test_metrics["aupr"]:.4f}'
    )

    fold_results.append({
        'fold': fold + 1,
        'best_epoch': best_epoch,
        'n_train': len(train_df),
        'n_val': len(val_df),
        'n_test': len(test_df),
        'n_train_drugs': len(train_drugs),
        'n_val_drugs': len(val_drugs),
        'n_test_drugs': len(test_drugs),
        'best_val_f1': best_val_metrics['f1'],
        'accuracy': test_metrics['accuracy'],
        'precision': test_metrics['precision'],
        'recall': test_metrics['recall'],
        'f1': test_metrics['f1'],
        'auc': test_metrics['auc'],
        'aupr': test_metrics['aupr']
    })

    all_test_labels.append(test_metrics['y_true'])
    all_test_probs.append(test_metrics['y_probs'])

    del model, optimizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()



OUTER FOLD 1/5
Train      : 454 samples / 43 drugs
Validation : 74 samples / 4 drugs
Outer Test : 94 samples / 12 drugs
Test drugs: ['citalopram', 'codeine', 'desmethyltamoxifen', 'dextromethorphan', 'donepezil', 'fluvoxamine', 'mianserin', 'mirtazapine', 'opipramol', 'timolol', 'tropisetron', 'zuclopenthixol']
Train class counts: {'de': 182, 'nor': 143, 'f': 129}
Val class counts  : {'de': 29, 'f': 29, 'nor': 16}
Test class counts : {'de': 39, 'f': 37, 'nor': 18}
Epoch [01/150] | Train Loss 1.2293 Acc 0.3921 | Val Loss 1.1167 Acc 0.2568 | Val Macro-F1 0.1841 AUC 0.7834 AUPR 0.6222
Epoch [02/150] | Train Loss 1.0774 Acc 0.4075 | Val Loss 1.0607 Acc 0.4595 | Val Macro-F1 0.3143 AUC 0.8082 AUPR 0.6653
Epoch [03/150] | Train Loss 1.0486 Acc 0.4361 | Val Loss 1.0546 Acc 0.4595 | Val Macro-F1 0.3143 AUC 0.7946 AUPR 0.6433
Epoch [04/150] | Train Loss 1.0298 Acc 0.4427 | Val Loss 1.0221 Acc 0.4730 | Val Macro-F1 0.4345 AUC 0.8059 AUPR 0.6493
Epoch [05/150] | Train Loss 1.0042 Acc 0.4736 | Va

In [32]:
# ================================
# 8. Fold-level results and mean ± sample SD
# ================================
results_df = pd.DataFrame(fold_results)
results_df.to_csv(
    'drug_grouped_genotype_only_fold_results.csv',
    index=False
)

assignments_df = pd.DataFrame(fold_assignments)
assignments_df.to_csv(
    'drug_grouped_genotype_only_fold_assignments.csv',
    index=False
)

metric_cols = [
    'accuracy', 'precision', 'recall',
    'f1', 'auc', 'aupr'
]

summary_rows = []
for metric in metric_cols:
    summary_rows.append({
        'metric': metric,
        'mean': results_df[metric].mean(),
        'sd': results_df[metric].std(ddof=1)
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(
    'drug_grouped_genotype_only_summary.csv',
    index=False
)

print('\nFold-level results')
print(results_df.to_string(index=False))

print('\nMean ± SD across outer folds')
for _, row in summary_df.iterrows():
    print(
        f'{row["metric"]}: '
        f'{row["mean"]:.4f} ± {row["sd"]:.4f}'
    )



Fold-level results
 fold  best_epoch  n_train  n_val  n_test  n_train_drugs  n_val_drugs  n_test_drugs  best_val_f1  accuracy  precision   recall       f1      auc     aupr
    1          63      454     74      94             43            4            12     0.811530  0.755319   0.748933 0.780242 0.748287 0.923749 0.856958
    2          60      425     78     119             42            6            11     0.891550  0.789916   0.786649 0.793468 0.789000 0.938906 0.889182
    3          10      394     73     155             39            8            12     0.684023  0.580645   0.689048 0.580317 0.574190 0.797670 0.663992
    4          52      431     83     108             42            6            11     0.739819  0.870370   0.878275 0.877924 0.873274 0.960707 0.911958
    5          29      396     80     146             41            5            13     0.761160  0.780822   0.793974 0.810618 0.796938 0.887587 0.819534

Mean ± SD across outer folds
accuracy: 0.7554 ± 0.1068
